# Ask the shop model

Type what a shopkeeper said on a call. The model tells you: **in stock**, **price**, **delivery** and
**other product offered**. The text can be Hindi, Marathi or Hinglish. It is not a chatbot.

**How to use it**
1. Open this notebook on kaggle.com → **Edit**.
2. Right panel:
   - **Input** shows `wimb-shop-model-training`. If not: **Add Input → Notebooks → Your Work →
     wimb-shop-model-training**.
   - **Accelerator: GPU T4**, **Internet: on**.
3. **Run all**. About 3–4 minutes each time you start it.
4. At the bottom: type the product and what the shopkeeper said, tap **Ask**. The answer comes in about
   half a second. Ask as many as you like. No box, only text like VBox(...)? Use cell 7, under it.
5. When done, stop the session (**Run → Stop session**, or the power icon), so your free GPU hours are not
   used up.

Tip: Accelerator **None** also works (about 10 s per answer) and uses no GPU hours.

In [ ]:
#@title 1. Helpers (no need to read; built from score.py, try_on_kaggle.py and Modelfile)
import os, shutil, subprocess, urllib.error, urllib.request  # json, re and time: score.py below

"""Shared by the notebook and the tests: the task's prompt, and how an answer is scored.

The task: read what a shopkeeper said on a phone call and return JSON with four fields.
"""

import json
import re
import time

SYSTEM = (
    "You read what a shopkeeper said on a phone call about a product. "
    "Reply with only a JSON object with these keys: "
    "in_stock (true, false or null), price_inr (a number or null), "
    "delivery (true, false or null), alternative (text or null). "
    "price_inr is the price of the product that was asked for. "
    "Use null when the shopkeeper did not say."
)

FIELDS = ("in_stock", "price_inr", "delivery", "alternative")


def user_message(product: str, said: str) -> str:
    return f"Product: {product}\nShopkeeper: {said}"


def answer_text(answer: dict) -> str:
    """The exact text the model is trained to reply with."""
    return json.dumps({k: answer[k] for k in FIELDS}, ensure_ascii=False)


def parse(text: str) -> dict | None:
    """The last JSON object in the reply that has exactly the four keys, or None.

    Every "{" is tried, so braces inside a value ("M-seal {2 pack}") or other text around the
    answer do not hide it.
    """
    decoder = json.JSONDecoder()
    found = None
    for match in re.finditer(r"\{", text):
        try:
            value, _ = decoder.raw_decode(text, match.start())
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict) and set(value) == set(FIELDS):
            found = value
    return found


def _tokens(text: str) -> set[str]:
    """Lower-case words, with "40mm" read as "40 mm" and punctuation dropped."""
    text = re.sub(r"(?<=\d)(?=[a-z])|(?<=[a-z])(?=\d)", " ", text.lower())
    return set(re.sub(r"[-_.,;:!?()\[\]{}\"'/]", " ", text).split())


def same(field: str, got, want) -> bool:
    if want is None or got is None:
        return got is None and want is None
    if field in ("in_stock", "delivery"):
        return isinstance(got, bool) and got == want  # 1 and 0 are not true and false
    if field == "price_inr":
        if isinstance(got, bool):
            return False
        try:
            return float(got) == float(want)
        except (TypeError, ValueError):
            return False
    # alternative: the wanted words must all be there, with at most two extra ("40 mm padlock").
    if not isinstance(got, str):
        return False
    have, need = _tokens(got), _tokens(want)
    return bool(have) and need <= have and len(have - need) <= 2


def score(replies: list[str], examples: list[dict]) -> dict:
    """Per-field accuracy, all-four-right accuracy, and the mistakes.

    A reply with no valid answer object (or a missing or extra key) gets every field wrong.
    """
    right = {f: 0 for f in FIELDS}
    all_right = json_ok = 0
    mistakes = []
    for reply, ex in zip(replies, examples, strict=True):
        got = parse(reply)
        if got is not None:
            json_ok += 1
        wrong = list(FIELDS) if got is None else [f for f in FIELDS if not same(f, got[f], ex["answer"][f])]
        for f in FIELDS:
            right[f] += f not in wrong
        if wrong:
            mistakes.append({"said": ex["said"], "want": ex["answer"], "reply": reply, "wrong": wrong})
        else:
            all_right += 1
    n = len(examples)
    return {
        "examples": n,
        "json_ok": json_ok / n,
        "all_correct": all_right / n,
        **{f: right[f] / n for f in FIELDS},
        "mistakes": mistakes,
    }


KEYS = ("all_correct", "json_ok", *FIELDS)


def evaluate(ask, sets: dict[str, list[dict]], deadline: float | None = None) -> dict:
    """Run ask(product, said) over every test set and score it. Every reply is kept.

    After `deadline` (a time.time() value) no more questions are asked; what was answered is
    scored, and "of" says how many there were.
    """
    out = {}
    for name, rows in sets.items():
        replies = []
        for ex in rows:
            if deadline is not None and time.time() > deadline:
                break
            replies.append(ask(ex["product"], ex["said"]))
        done = rows[: len(replies)]
        scored = score(replies, done) if done else {"examples": 0, "mistakes": []}
        out[name] = {**scored, "replies": replies, "of": len(rows)}
    return out


def show(label: str, results: dict, max_mistakes: int = 8) -> None:
    for name, res in results.items():
        n = res["examples"]
        if not n:
            print(f"== {label} [{name}]: no calls answered")
            continue
        partial = f" (only {n} of {res['of']} answered)" if n < res["of"] else ""
        print(f"== {label} [{name}]: all 4 fields right in {res['all_correct']:.0%} of {n} calls{partial}")
        print("   " + "   ".join(f"{k} {res[k]:.0%}" for k in KEYS[1:]))
        for m in res["mistakes"][:max_mistakes]:
            print("\n   said: ", m["said"])
            print("   want: ", answer_text(m["want"]))
            print("   reply:", m["reply"][:200])
            print("   wrong:", ", ".join(m["wrong"]))


def results_payload(model: str, epochs: int, before: dict, after: dict) -> dict:
    """What results.json holds: scores, mistakes and every raw reply, so scores can be recomputed."""
    sets = {}
    for name, a in after.items():
        b = before[name]
        sets[name] = {
            "examples": a["of"],
            "before": {k: round(b[k], 3) for k in KEYS},
            "after": {k: round(a[k], 3) for k in KEYS},
            "mistakes_before": b["mistakes"],
            "mistakes_after": a["mistakes"],
            "replies_before": b["replies"],
            "replies_after": a["replies"],
        }
    return {"model": model, "epochs": epochs, "sets": sets}


def make_modelfile(gguf_name: str) -> str:
    """An Ollama Modelfile whose template is exactly the text the model was trained on.

    Unsloth's own Qwen3 template differs a little (a blank line after the system message, and
    thinking-mode syntax that needs a newer Ollama), so it is not used.
    """
    return (
        f"FROM ./{gguf_name}\n"
        'TEMPLATE """{{ if .System }}<|im_start|>system\n{{ .System }}<|im_end|>\n{{ end }}'
        '{{ if .Prompt }}<|im_start|>user\n{{ .Prompt }}<|im_end|>\n{{ end }}'
        '<|im_start|>assistant\n{{ .Response }}<|im_end|>\n"""\n'
        'PARAMETER stop "<|im_end|>"\n'
        "PARAMETER temperature 0\n"
        # Calls are short (training used 1,024 tokens). Ollama's default context (32,768 on 0.40) takes about
        # 7.5 GB of memory with this model; 2,048 needs about 3 GB, so it fits a laptop with 8 GB.
        "PARAMETER num_ctx 2048\n"
        f'SYSTEM """{SYSTEM}"""\n'
    )


def in_words(answer):
    """An answer in plain words, e.g. "in stock: yes; price: ₹220; delivery: no; other product offered: none"."""
    if not isinstance(answer, dict):
        return "no answer (the reply was not the 4-field JSON)"

    def yes_no(value):
        if value is None:
            return "not said"
        return ("yes" if value else "no") if isinstance(value, bool) else repr(value)

    price = answer.get("price_inr")
    if price is None:
        price = "not said"
    elif isinstance(price, (int, float)) and not isinstance(price, bool):
        price = f"₹{int(price)}" if float(price).is_integer() else f"₹{price}"
    else:
        price = repr(price)
    alternative = answer.get("alternative")
    return (f"in stock: {yes_no(answer.get('in_stock'))}; price: {price}; delivery: {yes_no(answer.get('delivery'))}; "
            f"other product offered: {'none' if alternative is None else alternative}")


# The repo's Modelfile, to check the one made here is the same
MODELFILE_REPO = 'FROM ./wimb-shop-model.Q4_K_M.gguf\nTEMPLATE """{{ if .System }}<|im_start|>system\n{{ .System }}<|im_end|>\n{{ end }}{{ if .Prompt }}<|im_start|>user\n{{ .Prompt }}<|im_end|>\n{{ end }}<|im_start|>assistant\n{{ .Response }}<|im_end|>\n"""\nPARAMETER stop "<|im_end|>"\nPARAMETER temperature 0\nPARAMETER num_ctx 2048\nSYSTEM """You read what a shopkeeper said on a phone call about a product. Reply with only a JSON object with these keys: in_stock (true, false or null), price_inr (a number or null), delivery (true, false or null), alternative (text or null). price_inr is the price of the product that was asked for. Use null when the shopkeeper did not say."""\n'

WORKING_DIR = "/kaggle/working"  # ask_check.json is saved here (the notebook's Output tab)
RESULTS = {"ollama_version": None, "gpu": None, "model_file": None, "model_mb": None, "create_seconds": None,
           "load_seconds": None, "processor": None, "self_check": None, "ask_box": None, "errors": []}
OLLAMA_LOG = "/tmp/ollama.log"  # what `ollama serve` prints (cell 3)


def save_results():
    """Write RESULTS to ask_check.json. Done after every stage, so a stop part-way loses nothing."""
    if os.path.isdir(WORKING_DIR):
        with open(os.path.join(WORKING_DIR, "ask_check.json"), "w", encoding="utf-8") as fh:
            json.dump(RESULTS, fh, ensure_ascii=False, indent=1)


def failed(stage, error):
    """Record a failed stage and go on (Kaggle's Run all stops at the first error that is not caught)."""
    message = f"{stage}: {error!r}" if isinstance(error, Exception) else f"{stage}: {error}"
    print("FAILED", message)
    RESULTS["errors"].append(message)
    save_results()


def sh(command, timeout=120):
    """What a shell command printed, or "" when the command is missing or fails."""
    try:
        return subprocess.run(command, shell=True, capture_output=True, text=True, timeout=timeout).stdout.strip()
    except Exception:
        return ""


def generate(body):
    request = urllib.request.Request(OLLAMA_URL + "/api/generate", json.dumps(body).encode(),
                                     {"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=600) as resp:
        return json.load(resp)


def in_memory():
    """The model as Ollama holds it now (from /api/ps: size, size_vram, ...), or None when it is not loaded."""
    with urllib.request.urlopen(OLLAMA_URL + "/api/ps", timeout=10) as resp:
        models = json.load(resp).get("models") or []
    return next((m for m in models if m.get("name", "").split(":")[0] == "wimb-shop"), None)


def processor():
    """Where the model runs now, as `ollama ps` says it: "100% GPU", "100% CPU", "25%/75% CPU/GPU", or None."""
    try:
        m = in_memory()
    except Exception:
        return None
    if not m or not m.get("size"):
        return None
    cpu = round(100 * (m["size"] - (m.get("size_vram") or 0)) / m["size"])
    return "100% GPU" if cpu <= 0 else "100% CPU" if cpu >= 100 else f"{cpu}%/{100 - cpu}% CPU/GPU"


NOT_RUNNING = "The model is not running (the session may have restarted): tap Run all again"
SAVED_RUN = "This is the saved check, where nobody can type. To ask questions: tap Edit, then Run all."
NO_BOX = "No box, only text like VBox(...)? Use cell 7, under it."


def saved_run():
    """True in Kaggle's saved run (Save Version, or try_on_kaggle.py --make-ask-notebook): no one can type there."""
    return os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch"  # "Interactive" in the editor


def ask(product, said):
    """The model's reply and the seconds it took, through Ollama's API as an app would. No system message is
    sent (the Modelfile has it); keep_alive keeps the model loaded for 60 minutes after each question."""
    body = {"model": "wimb-shop", "prompt": user_message(product, said), "stream": False, "keep_alive": "60m",
            "options": {"temperature": 0, "num_predict": 96}}
    started = time.time()
    try:
        data = generate(body)
    except urllib.error.HTTPError as e:  # Ollama runs but refused, e.g. the model was not made
        raise RuntimeError(f"Ollama said: {e.read().decode(errors='replace')[:300]} (tap Run all again)") from e
    except (urllib.error.URLError, ConnectionError) as e:  # nothing answers: Ollama is not running
        raise RuntimeError(NOT_RUNNING) from e
    return data.get("response", "").strip(), round(time.time() - started, 2)


def where_it_runs(ps, gpu):
    """Where the model runs, in words. ps: what processor() gave; gpu: the GPU's name (None: no GPU)."""
    if ps == "100% GPU":
        return "running on the GPU: about half a second per answer"
    if ps == "100% CPU":
        return "running on CPU only: about 10 s per answer" + (" (Ollama did not use the GPU)" if gpu else "")
    return f"running on GPU and CPU ({ps})" if ps else "running on GPU or CPU (not known)"


def answer_lines(product, said, reply, seconds):
    """One question and its answer as lines for a person: the answer in plain words, the model's JSON, the time."""
    return [f"Product: {product}", f"Shopkeeper: {said}", f"Answer: {in_words(parse(reply))}",
            f"Model's reply: {reply}", f"Took {seconds} s"]


def show_answer(product, said):
    """Ask, then print the answer (or why it could not ask)."""
    try:
        reply, seconds = ask(product, said)
    except Exception as e:
        print("Could not ask:", str(e) or repr(e))
        return
    print("\n".join(answer_lines(product, said, reply, seconds)))


def make_ask_box(ask_fn):
    """The ask box: Product, Shopkeeper said, an Ask button, and the answers so far (newest first).

    ask_fn(product, said) gives (reply, seconds). Its errors are shown in the box (ipywidgets would hide them).
    """
    import html
    import ipywidgets as widgets

    product = widgets.Text(placeholder="e.g. 16A MCB", layout=widgets.Layout(width="100%"))
    said = widgets.Textarea(placeholder="e.g. haan hai, 220 ka, delivery nahi\nor: nahi hai, 32A wala le lo",
                            rows=3, layout=widgets.Layout(width="100%"))
    button = widgets.Button(description="Ask", button_style="primary")
    status = widgets.Label()
    history = widgets.HTML()
    shown = []  # the answers so far, newest first

    def on_click(_):
        question = (product.value.strip(), said.value.strip())
        if not question[1]:
            status.value = "Type what the shopkeeper said, then tap Ask."
            return
        button.disabled, status.value = True, "Asking…"
        try:
            reply, seconds = ask_fn(*question)
            lines = answer_lines(*question, reply, seconds)
            said.value = ""  # ready for the next question; the product stays
        except Exception as e:
            lines = [f"Product: {question[0]}", f"Shopkeeper: {question[1]}", f"Could not ask: {str(e) or repr(e)}"]
        finally:
            button.disabled, status.value = False, ""
        shown.insert(0, "\n".join(lines))
        history.value = "".join(f"<pre style='white-space: pre-wrap'>{html.escape(t)}</pre>" for t in shown)

    button.on_click(on_click)
    return widgets.VBox([widgets.Label("Product"), product, widgets.Label("Shopkeeper said"), said, button, status,
                         history])


def show_ask_box():
    """Show the ask box (under the cell that calls this). Without ipywidgets, say to use the next cell.

    In Kaggle's saved run the box is only built (to check that it can be), not shown: no one can type there.
    """
    try:
        import ipywidgets
        from IPython.display import display
    except ImportError:
        RESULTS["ask_box"] = "no ipywidgets"
        print("The ask box needs ipywidgets, which is missing here: use the next cell instead.")
    else:
        try:
            box = make_ask_box(ask)
            if saved_run():
                RESULTS["ask_box"] = f"built, not shown (Kaggle's saved run; ipywidgets {ipywidgets.__version__})"
                print(SAVED_RUN)
            else:
                print(NO_BOX)  # the box itself is drawn by the editor: this check can't see it
                display(box)
                RESULTS["ask_box"] = f"shown (ipywidgets {ipywidgets.__version__})"
        except Exception as e:
            RESULTS["ask_box"] = "failed"
            failed("ask box", e)
            print("Use the next cell instead.")
    save_results()

In [ ]:
#@title 2. Find the trained model (the training notebook's output, attached as input)
import glob


def rank(path):
    name = os.path.basename(path).lower()
    return ("wimb-shop-model" not in name or "q4_k_m" not in name, "q4_k_m" not in name, path)


found = sorted(glob.glob("/kaggle/input/**/*.gguf", recursive=True), key=rank)
if not found:
    print("No .gguf file under /kaggle/input. What is attached:")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root[len("/kaggle/input"):].count(os.sep)
        print("   " * depth + (os.path.basename(root) or root) + "/")
        for name in sorted(files)[:20]:
            print("   " * (depth + 1) + name)
        if depth >= 3:
            dirs[:] = []
    failed("find the model", "no .gguf file under /kaggle/input")
    raise RuntimeError("No model file. Attach the training notebook's output: Add Input → Notebooks → Your Work → "
                       "wimb-shop-model-training, then Run all again.")
MODEL_FILE = found[0]
RESULTS["model_file"] = MODEL_FILE
RESULTS["model_mb"] = round(os.path.getsize(MODEL_FILE) / 1e6)
print(f"Model file: {MODEL_FILE}  ({RESULTS['model_mb']:,} MB)")
save_results()

In [ ]:
#@title 3. Install Ollama and start it (about 1–2 minutes)
os.environ["OLLAMA_KEEP_ALIVE"] = "60m"  # keep the model loaded between questions

OLLAMA_URL = "http://127.0.0.1:11434"


def ollama_version():
    try:
        with urllib.request.urlopen(OLLAMA_URL + "/api/version", timeout=5) as resp:
            return json.load(resp).get("version")
    except Exception:
        return None


if not ollama_version():
    if not shutil.which("ollama"):
        # zstd unpacks Ollama's download (.tar.zst). The download already holds the CUDA libraries; lspci is
        # left out on purpose: with it (and no nvidia-smi) install.sh would try to install NVIDIA drivers.
        if not shutil.which("zstd"):
            sh("apt-get update -qq; DEBIAN_FRONTEND=noninteractive apt-get install -y -qq zstd", timeout=600)
        install = subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True,
                                 capture_output=True, text=True, timeout=1200)
        print("\n".join((install.stdout + install.stderr).replace("\r", "\n").splitlines()[-15:]))
        if not shutil.which("ollama"):
            failed("install Ollama", (install.stdout + install.stderr)[-1500:])
            raise RuntimeError("Ollama did not install: check Internet is on (right panel → Session options), then Run all again")
    print(sh("ollama --version 2>&1"))
    log = open(OLLAMA_LOG, "w")  # stays open: the server writes to it
    server = subprocess.Popen(["ollama", "serve"], env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"},
                              stdout=log, stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_version() or server.poll() is not None:
            break
        time.sleep(1)
if not ollama_version():
    tail = sh(f"tail -n 40 {OLLAMA_LOG}")
    print(tail)
    failed("start Ollama", tail[-1500:])
    raise RuntimeError("Ollama did not start (its log is above)")
RESULTS["ollama_version"] = ollama_version()
RESULTS["gpu"] = (sh("nvidia-smi --query-gpu=name --format=csv,noheader").splitlines() or [None])[0]
print("Ollama", RESULTS["ollama_version"], "is running. GPU:", RESULTS["gpu"])
save_results()

In [ ]:
#@title 4. Make the Ollama model from the file, exactly as on a PC (ollama create, about 1 minute)
os.makedirs("/tmp/wimb", exist_ok=True)
link = "/tmp/wimb/wimb-shop-model.Q4_K_M.gguf"
if os.path.lexists(link):
    os.remove(link)
os.symlink(MODEL_FILE, link)
modelfile = make_modelfile("wimb-shop-model.Q4_K_M.gguf")
if modelfile != MODELFILE_REPO:
    failed("Modelfile", "the Modelfile made here differs from the repo's Modelfile")
    raise RuntimeError("The Modelfile made here differs from the repo's Modelfile: run build_notebook.py again")
with open("/tmp/wimb/Modelfile", "w", encoding="utf-8") as fh:
    fh.write(modelfile)
print(modelfile)


def create():
    started = time.time()
    done = subprocess.run(["ollama", "create", "wimb-shop", "-f", "/tmp/wimb/Modelfile"],
                          capture_output=True, text=True, timeout=1800)
    output = re.sub(r"\x1b\[[0-9;?]*[A-Za-z]", "", done.stdout + done.stderr).replace("\r", "\n")  # no progress codes
    print("\n".join(line for line in output.splitlines()[-8:] if line.strip()))
    return done.returncode, output, round(time.time() - started, 1)


exit_code, output, RESULTS["create_seconds"] = create()
if exit_code != 0:  # some versions do not follow the link: try once with a real copy
    print("ollama create failed with the linked file; trying again with a copy")
    os.remove(link)
    shutil.copy(MODEL_FILE, link)
    exit_code, output, RESULTS["create_seconds"] = create()
if exit_code != 0:
    failed("ollama create", output[-1500:])
    raise RuntimeError("ollama create failed (see above)")
print(f"ollama create took {RESULTS['create_seconds']} s")
save_results()

In [ ]:
#@title 5. Load the model and check that it answers (about 1 minute the first time)
CHECK = {"product": "16A MCB", "said": "haan hai, 220 ka, delivery nahi"}
try:
    started = time.time()
    generate({"model": "wimb-shop", "keep_alive": "60m"})  # no prompt: only load it
    RESULTS["load_seconds"] = round(time.time() - started, 1)
    print(f"Model loaded in {RESULTS['load_seconds']} s")
except Exception as e:
    failed("load the model", e)
RESULTS["self_check"] = None
try:
    reply, seconds = ask(CHECK["product"], CHECK["said"])
    RESULTS["self_check"] = {**CHECK, "reply": reply, "parsed": parse(reply), "seconds": seconds}
    print("\n".join(answer_lines(CHECK["product"], CHECK["said"], reply, seconds)))
except Exception as e:
    failed("self-check", e)
RESULTS["processor"] = processor()
if RESULTS["self_check"] is None:
    print("\nThe model is NOT running (see FAILED above): check the right panel (Input, Accelerator, Internet) "
          "and tap Run all again")
else:
    print("\nThe model is", where_it_runs(RESULTS["processor"], RESULTS["gpu"]))
if RESULTS["processor"] == "100% CPU" and RESULTS["gpu"]:
    failed("GPU", f"there is a GPU ({RESULTS['gpu']}) but Ollama runs the model on the CPU")
if RESULTS["self_check"] and RESULTS["self_check"]["parsed"]:
    print(SAVED_RUN if saved_run() else "Ready: ask your questions in the box below.")
save_results()

In [ ]:
#@title 6. Ask here: type the product and what the shopkeeper said, then tap Ask
if "OLLAMA_URL" in globals() and "show_ask_box" in globals():
    show_ask_box()
else:  # a new session (Kaggle stopped or restarted it): nothing above has run in it
    print("The session restarted: tap Run all again (about 3–4 minutes)")

In [ ]:
#@title 7. Or type here: change the two lines, then run this cell again (Shift+Enter, or the ▶ button)
PRODUCT = "16A MCB"
SAID = "haan hai, 220 ka, delivery nahi"
if "OLLAMA_URL" in globals() and "show_answer" in globals():
    show_answer(PRODUCT, SAID)
else:  # a new session (Kaggle stopped or restarted it): nothing above has run in it
    print("The session restarted: tap Run all again (about 3–4 minutes)")

## Tips
- No box, only text like VBox(...)? Use cell 7, under it.
- One product per question.
- Write what the shopkeeper said as text, the way they said it (Hindi, Marathi or Hinglish).
- It answers four things only: in stock, price, delivery, other product offered. It is not a chatbot.
- If it says the model is not running, or the session has stopped: **Run all** again.
- When done, stop the session (**Run → Stop session**), so your free GPU hours are not used up.